# Multi-hop RAG with Iterative Retrieval

This notebook implements a lightweight sequential retrieval-augmented generation (RAG) pipeline for multi-hop question answering.

At each hop, the system retrieves relevant passages, extracts an intermediate finding, and checks whether the accumulated evidence is sufficient to answer the original question. If the evidence is insufficient, the language model generates another retrieval query. The process stops when the evidence is judged sufficient or the maximum number of hops is reached.

**Experiment setup**
- Dataset: HotpotQA
- Evaluation set: a fixed subset of 30 questions
- Embedding model: `sentence-transformers/all-MiniLM-L6-v2`
- Vector index: the prebuilt FAISS index
- Language model: the Qwen model exposed through `qwen_model.py`

The notebook reuses the corpus, embeddings index, evaluation subset, and model wrapper created for the other RAG experiments. It does not build the corpus or index from scratch.


## 1. Environment Setup

Install the required packages, mount Google Drive, and define the project paths.

In [ ]:
%pip install -q transformers accelerate bitsandbytes sentence-transformers faiss-cpu


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 19.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 60.2 MB/s eta 0:00:00


In [ ]:
from pathlib import Path
from google.colab import drive

# Mount Google Drive
drive.mount("/content/drive")

# Define the project root directory
PROJECT_ROOT = Path("/content/drive/MyDrive/Multi_hop_RAG")

# Define project directories
DATA_DIR = PROJECT_ROOT / "data"
CORPUS_DIR = PROJECT_ROOT / "corpus"
INDEX_DIR = PROJECT_ROOT / "indexes"
RESULTS_DIR = PROJECT_ROOT / "results"

# Create the results directory if it does not exist
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# Display project paths
print(f"Project root: {PROJECT_ROOT}")
print(f"Data directory: {DATA_DIR}")
print(f"Corpus directory: {CORPUS_DIR}")
print(f"Index directory: {INDEX_DIR}")
print(f"Results directory: {RESULTS_DIR}")

# Check whether the main project directory exists
assert PROJECT_ROOT.exists(), (
    f"Project directory not found: {PROJECT_ROOT}"
)

print("\nProject directory verified successfully.")


Mounted at /content/drive
Project root: /content/drive/MyDrive/Multi_hop_RAG
Data directory: /content/drive/MyDrive/Multi_hop_RAG/data
Corpus directory: /content/drive/MyDrive/Multi_hop_RAG/corpus
Index directory: /content/drive/MyDrive/Multi_hop_RAG/indexes
Results directory: /content/drive/MyDrive/Multi_hop_RAG/results

Project directory verified successfully.


## 2. Load the Language Model

In [ ]:
import sys

MODEL_MODULE_DIR = Path("/content/drive/MyDrive")

if str(MODEL_MODULE_DIR) not in sys.path:
    sys.path.insert(0, str(MODEL_MODULE_DIR))

# Import the existing LLM interface
from qwen_model import call_llm

print("LLM interface imported successfully.")


Loading Qwen tokenizer...
Model path: /content/drive/MyDrive/Model_qwen2.5-7B-instruct/models--Qwen--Qwen2.5-7B-Instruct/snapshots/a09a35458c702b33eeacc393d103063234e8bc28


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading Qwen2.5-7B-Instruct in 8-bit...


Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

Qwen model loaded successfully.
Device: cuda:0
VRAM allocated: 8.71 GB
LLM interface imported successfully.


## 3. Load embedding model

Model: `sentence-transformers/all-MiniLM-L6-v2`

In [ ]:
from sentence_transformers import SentenceTransformer

EMBEDDING_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"

embedding_model = SentenceTransformer(
    EMBEDDING_MODEL_NAME
)

print("✓ Embedding model loaded.")
print(f"Model: {EMBEDDING_MODEL_NAME}")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

✓ Embedding model loaded.
Model: sentence-transformers/all-MiniLM-L6-v2


## 4. Load the Corpus and FAISS Index

Load the existing corpus and index. The consistency check ensures that each corpus chunk corresponds to an indexed vector.



In [ ]:
import json
import faiss
import os
import numpy as np

DATASET_NAME = "hotpotqa"
N_SAMPLES = 7405
N_EVAL = 30

# Define file paths
corpus_path = f"{PROJECT_ROOT}/corpus/{DATASET_NAME}/corpus.json"
index_path = f"{PROJECT_ROOT}/indexes/{DATASET_NAME}/faiss.index"

# Load corpus
assert os.path.exists(corpus_path), (
    f"Corpus file not found: {corpus_path}"
)

with open(corpus_path, "r", encoding="utf-8") as f:
    corpus = json.load(f)

print(f"Corpus loaded successfully: {len(corpus)} chunks")

# Load FAISS index
assert os.path.exists(index_path), (
    f"FAISS index file not found: {index_path}"
)

index = faiss.read_index(index_path)

print("FAISS index loaded successfully.")
print(f"Index type       : {type(index).__name__}")
print(f"Embedding dim    : {index.d}")
print(f"Number of vectors: {index.ntotal}")

# Verify corpus and index consistency
assert len(corpus) == index.ntotal, (
    f"Corpus/index mismatch: {len(corpus)} chunks "
    f"but {index.ntotal} indexed vectors"
)

print("\nCorpus and FAISS index are ready.")


Corpus loaded successfully: 73700 chunks
FAISS index loaded successfully.
Index type       : IndexFlatIP
Embedding dim    : 384
Number of vectors: 73700

Corpus and FAISS index are ready.


## 5. Load the Evaluation Set

Use the same 30 questions across the RAG experiments to support a fair comparison. This is a small evaluation subset intended for experimentation, not a full benchmark evaluation.


In [ ]:
# Define the evaluation file path
evaluation_path = f"{DATA_DIR}/{DATASET_NAME}/evaluation_30.json"

# Verify that the evaluation file exists
assert os.path.exists(evaluation_path), (
    f"Evaluation file not found: {evaluation_path}"
)

# Load evaluation samples
with open(evaluation_path, "r", encoding="utf-8") as f:
    evaluation_samples = json.load(f)

print(f"Evaluation samples loaded: {len(evaluation_samples)}")

# Verify the evaluation dataset
assert len(evaluation_samples) == N_EVAL, (
    f"Expected {N_EVAL} evaluation samples, "
    f"but found {len(evaluation_samples)}"
)

# Display one example
print("\nExample evaluation question:")
print(evaluation_samples[0]["question"])
print("\nExpected answer:")
print(evaluation_samples[0]["answer"])


Evaluation samples loaded: 30

Example evaluation question:
What was Iqbal F. Qadir on when he participated in an attack on a radar station located on western shore of the Okhamandal Peninsula?

Expected answer:
flotilla


## 6. Configure Iterative Retrieval

`MAX_HOPS` limits the number of retrieval rounds. `TOP_K` controls how many unique chunks are returned for each query.


In [ ]:
# Sequential RAG configuration

MAX_HOPS = 3
TOP_K = 5

print("Sequential RAG configuration")
print(f"Maximum hops: {MAX_HOPS}")
print(f"Top-K retrieval: {TOP_K}")


Sequential RAG configuration
Maximum hops: 3
Top-K retrieval: 5


## 7. Implement the Retriever

The retriever embeds each query, searches the FAISS index, and removes exact duplicate chunk text from the candidate results.


In [ ]:
# Sequential RAG: retrieval function

def retrieve_chunks(query, top_k=5):
    query_embedding = embedding_model.encode(
        [query],
        convert_to_numpy=True,
        normalize_embeddings=True
    ).astype(np.float32)

    # Retrieve more candidates so duplicates do not reduce
    # the number of unique results.
    candidate_k = min(top_k * 3, index.ntotal)

    scores, indices = index.search(
        query_embedding,
        candidate_k
    )

    results = []
    seen_texts = set()

    for score, idx in zip(scores[0], indices[0]):
        if idx < 0:
            continue

        chunk = corpus[int(idx)]

        # Normalize whitespace before checking duplicate text.
        normalized_text = " ".join(
            chunk["text"].split()
        ).casefold()

        if normalized_text in seen_texts:
            continue

        seen_texts.add(normalized_text)

        results.append({
            "chunk_id": chunk["chunk_id"],
            "sample_id": chunk["sample_id"],
            "title": chunk["title"],
            "text": chunk["text"],
            "score": float(score),
            "dataset": chunk["dataset"],
            "source_type": chunk["source_type"]
        })

        if len(results) >= top_k:
            break

    return results


print("Retrieval function defined successfully.")


Retrieval function defined successfully.


## 8. Define Prompts and Parse Intermediate Responses

The system uses one prompt to generate the next retrieval query and another to extract a finding and assess whether the evidence answers the original question.


In [ ]:
def build_next_query_prompt(question, history_text):
    """
    Generate a focused next retrieval query from the original
    question and the findings collected so far.
    """

    return f"""
You are a search query generator for a multi-hop question answering system.

Original question:
{question}

Previous findings:
{history_text}

Identify the specific missing fact needed to answer the original question.
Generate one short search query targeting that missing fact.

Rules:
- If the findings say INSUFFICIENT EVIDENCE, continue searching.
- Use relevant names, locations, events, and relationships.
- Do not copy the previous history or write explanations.
- Do not include labels such as "SEARCH QUERY:" or "Hop 2 query:".
- Do not answer the original question.
- Output only the search query, preferably one concise sentence.
- Output STOP only if the original question can already be answered
  from the findings.
""".strip()



print("\nPrompt builders defined successfully.")



Prompt builders defined successfully.


In [ ]:
def build_intermediate_answer_prompt(
    question,
    query,
    evidence_text,
    history_text
):
    """
    Extract an intermediate finding and decide whether the
    collected evidence is sufficient to answer the original question.
    """

    return f"""
You are an evidence-based assistant in a sequential retrieval system.

Original question:
{question}

Current search query:
{query}

Findings from previous steps:
{history_text}

Documents retrieved in the current step:
{evidence_text}

Tasks:
1. Extract the most relevant finding supported by the documents.
2. Consider both the current documents and previous findings.
3. Decide whether the collected evidence is sufficient to answer
   the original question.

Return exactly these three fields:

FINDING: <a concise finding supported by the evidence, or INSUFFICIENT EVIDENCE>

SUFFICIENT: <YES or NO>

REASON: <one short sentence explaining the decision>

Rules:
- Use only information supported by the supplied documents.
- SUFFICIENT: YES only if the evidence supports a direct answer
  to the original question.
- If an important fact is missing, set SUFFICIENT: NO.
- Do not treat a plausible guess as evidence.
- Do not repeat the entire document text.
- Do not add any other fields.
""".strip()

print("Intermediate-answer prompt defined successfully.")

Intermediate-answer prompt defined successfully.


In [ ]:
import re

def parse_intermediate_response(response):
    """
    Parse FINDING, SUFFICIENT, and REASON from the LLM response.
    """

    finding_match = re.search(
        r"FINDING:\s*(.*?)(?=\nSUFFICIENT:|\Z)",
        response,
        flags=re.IGNORECASE | re.DOTALL
    )

    sufficient_match = re.search(
        r"SUFFICIENT:\s*(YES|NO)",
        response,
        flags=re.IGNORECASE
    )

    reason_match = re.search(
        r"REASON:\s*(.*)",
        response,
        flags=re.IGNORECASE | re.DOTALL
    )

    finding = (
        finding_match.group(1).strip()
        if finding_match else response.strip()
    )

    sufficient = (
        sufficient_match.group(1).upper() == "YES"
        if sufficient_match else False
    )

    reason = (
        reason_match.group(1).strip()
        if reason_match else "Could not parse the reason."
    )

    return {
        "finding": finding,
        "sufficient": sufficient,
        "reason": reason
    }


# Test the parser with a sample response
test_response = """
FINDING: Tivoli Gardens opened on 15 August 1843.

SUFFICIENT: YES

REASON: The text directly states the opening date.
"""

print(parse_intermediate_response(test_response))


{'finding': 'Tivoli Gardens opened on 15 August 1843.', 'sufficient': True, 'reason': 'The text directly states the opening date.'}


## 9. Implement the Iterative Retrieval Loop

For each hop, the system retrieves passages, asks the model to summarize the relevant finding and judge evidence sufficiency, and—if necessary—generates the next query. The loop also records a trace for later inspection.


In [ ]:
# Sequential RAG: main retrieval loop

def run_sequential_retrieval(question, max_hops=3, top_k=5):
    history = []
    trace = []
    used_queries = set()
    seen_chunk_ids = set()
    current_query = question
    stop_reason = "maximum_hops_reached"

    for hop in range(1, max_hops + 1):
        normalized_query = current_query.strip().lower()

        if normalized_query in used_queries:
            stop_reason = "repeated_query"
            break

        used_queries.add(normalized_query)

        # 1. Retrieve relevant chunks
        retrieved_chunks = retrieve_chunks(
            current_query,
            top_k=top_k
        )

        # 2. Remove duplicate chunks within the current retrieval
        unique_chunks = []
        current_chunk_ids = set()

        for chunk in retrieved_chunks:
            chunk_id = chunk["chunk_id"]

            if chunk_id not in current_chunk_ids:
                unique_chunks.append(chunk)
                current_chunk_ids.add(chunk_id)

        retrieved_chunks = unique_chunks

        evidence_text = "\n\n".join(
            f"Title: {chunk['title']}\n"
            f"Chunk ID: {chunk['chunk_id']}\n"
            f"Text: {chunk['text']}"
            for chunk in retrieved_chunks
        )

        history_text = (
            "\n".join(
                f"Hop {item['hop']} query: {item['query']}\n"
                f"Finding: {item['finding']}"
                for item in history
            )
            if history
            else "No previous findings."
        )

        # 3. Generate an intermediate finding and assess sufficiency
        answer_prompt = build_intermediate_answer_prompt(
            question=question,
            query=current_query,
            evidence_text=evidence_text,
            history_text=history_text
        )

        raw_response = call_llm(
            answer_prompt,
            max_tokens=160
        ).strip()

        parsed = parse_intermediate_response(raw_response)

        finding = parsed["finding"]
        sufficient = parsed["sufficient"]
        reason = parsed["reason"]

        # 4. Record this hop
        trace.append({
            "hop": hop,
            "query": current_query,
            "retrieved_chunks": retrieved_chunks,
            "finding": finding,
            "sufficient": sufficient,
            "reason": reason,
            "raw_response": raw_response
        })

        history.append({
            "hop": hop,
            "query": current_query,
            "finding": finding,
            "sufficient": sufficient,
            "reason": reason
        })

        for chunk in retrieved_chunks:
            seen_chunk_ids.add(chunk["chunk_id"])

        # 5. Stop if the evidence is sufficient
        if sufficient:
            stop_reason = "sufficient_evidence"
            break

        # 6. If no sufficient answer was found, check the hop limit
        if hop == max_hops:
            stop_reason = "maximum_hops_reached"
            break

        # 7. Generate the next retrieval query
        updated_history_text = (
            history_text
            + "\n\n"
            + f"Latest retrieved evidence:\n{evidence_text}"
            + "\n\n"
            + f"Latest finding:\n{finding}"
            + "\n"
            + f"Latest sufficiency decision: "
              f"{'YES' if sufficient else 'NO'}"
            + "\n"
            + f"Reason: {reason}"
        )

        next_query_prompt = build_next_query_prompt(
            question=question,
            history_text=updated_history_text
        )

        next_query = call_llm(
            next_query_prompt,
            max_tokens=80
        ).strip()

        # Remove common formatting artifacts
        next_query = next_query.strip("\"'` ")
        next_query = next_query.removeprefix(
            "SEARCH QUERY:"
        ).strip()
        next_query = next_query.removeprefix(
            "Search query:"
        ).strip()
        next_query = next_query.removeprefix(
            "Query:"
        ).strip()

        if next_query.upper() == "STOP":
            stop_reason = "model_stopped"
            break

        if not next_query:
            stop_reason = "empty_query"
            break

        if next_query.lower() in used_queries:
            stop_reason = "repeated_query"
            break

        current_query = next_query

    return {
        "question": question,
        "history": history,
        "trace": trace,
        "stop_reason": stop_reason,
        "num_hops": len(trace),
        "num_retrievals": len(trace),
        "num_llm_calls": len(trace) + max(0, len(trace) - 1),
        "unique_chunks_retrieved": len(seen_chunk_ids)
    }



print("Sequential retrieval loop defined successfully.")


Sequential retrieval loop defined successfully.


## 10. Generate the Final Answer

After retrieval stops, provide the accumulated evidence and intermediate findings to the model to generate a concise answer to the original question.


In [ ]:

def build_final_answer_prompt(
    question,
    evidence_text,
    intermediate_findings
):
    return f"""
You are an evidence-based question answering assistant.

Answer the ORIGINAL QUESTION using the collected evidence.

ORIGINAL QUESTION:
{question}

INTERMEDIATE FINDINGS:
{intermediate_findings}

COLLECTED EVIDENCE:
{evidence_text}

Instructions:
1. Identify exactly what entity, property, or relationship
   the original question asks about.
2. Select evidence that directly supports that requested
   information. A passage about a related entity is not
   sufficient if it answers a different question.
3. Distinguish facts about related entities.
4. Do not choose an answer merely because it appears in
   a highly relevant passage. Verify that it answers the
   specific question asked.
5. Use only the collected evidence. Do not guess or use
   outside knowledge.
6. If the evidence does not support a reliable answer,
   state that it cannot be determined from the evidence.

Return only the concise final answer, without explanation.
"""


In [ ]:
def generate_final_answer(question, retrieval_result):
    """
    Generate a final answer from the evidence collected
    during sequential retrieval.
    """

    # 1. Collect unique evidence across all retrieval hops.
    collected_chunks = []
    seen_texts = set()

    for hop_data in retrieval_result["trace"]:
        for chunk in hop_data["retrieved_chunks"]:
            normalized_text = " ".join(
                chunk["text"].split()
            ).casefold()

            if normalized_text in seen_texts:
                continue

            seen_texts.add(normalized_text)
            collected_chunks.append(chunk)

    # 2. Format the collected evidence.
    if collected_chunks:
        evidence_parts = []

        for i, chunk in enumerate(collected_chunks, start=1):
            evidence_parts.append(
                f"[Evidence {i}]\n"
                f"Title: {chunk['title']}\n"
                f"Text: {chunk['text']}"
            )

        evidence_text = "\n\n".join(evidence_parts)
    else:
        evidence_text = "No evidence was retrieved."

    # 3. Collect intermediate findings.
    intermediate_findings = "\n".join(
        f"Hop {item['hop']}: {item['finding']}"
        for item in retrieval_result["history"]
    )

    if not intermediate_findings:
        intermediate_findings = "No intermediate findings."

    # 4. Build the final-answer prompt.
    final_prompt = build_final_answer_prompt(
        question=question,
        evidence_text=evidence_text,
        intermediate_findings=intermediate_findings
    )

    # 5. Generate the final answer.
    final_answer = call_llm(
        final_prompt,
        max_tokens=160
    ).strip()

    # 6. Return the answer and useful execution metadata.
    return {
        "question": question,
        "final_answer": final_answer,
        "num_evidence_chunks": len(collected_chunks),
        "retrieval_stop_reason": retrieval_result["stop_reason"],
        "num_hops": retrieval_result["num_hops"],
        "num_retrievals": retrieval_result["num_retrievals"],
        "num_llm_calls": retrieval_result["num_llm_calls"] + 1
    }


## 11. Run the Evaluation

Run the full 30-question evaluation. Each question is processed independently, and the results file is updated after every question so progress is preserved if the session is interrupted.

This step makes LLM calls and may take several minutes.


In [ ]:
import json
import time
import os
from pathlib import Path

# 1. Configure the output file
OUTPUT_DIR = Path(RESULTS_DIR) / DATASET_NAME / "results"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

RESULTS_PATH = OUTPUT_DIR / "debugging_results_30_iterative_retrieval.json"

# 2. Initialize results
debugging_results = []
total_samples = len(evaluation_samples)

print(f"Starting debugging experiment")
print(f"Total questions: {total_samples}")
print(f"Max hops: {MAX_HOPS}")
print(f"Top-k: {TOP_K}")
print(f"Results will be saved to: {RESULTS_PATH}")

# 3. Run each question independently
for i, sample in enumerate(evaluation_samples, start=1):
    question = sample["question"]
    start_time = time.time()

    print(f"\n[{i}/{total_samples}] {question}")

    try:
        # Run sequential retrieval
        retrieval_result = run_sequential_retrieval(
            question=question,
            max_hops=MAX_HOPS,
            top_k=TOP_K
        )

        # Generate the final answer
        final_result = generate_final_answer(
            question=question,
            retrieval_result=retrieval_result
        )

        elapsed_time = time.time() - start_time

        result = {
            "sample_index": i,
            "question": question,
            "status": "success",
            "final_answer": final_result["final_answer"],
            "num_evidence_chunks": final_result["num_evidence_chunks"],
            "stop_reason": final_result["retrieval_stop_reason"],
            "num_hops": final_result["num_hops"],
            "num_retrievals": final_result["num_retrievals"],
            "num_llm_calls": final_result["num_llm_calls"],
            "elapsed_seconds": round(elapsed_time, 2),
            "retrieval_trace": retrieval_result["trace"],
            "intermediate_history": retrieval_result["history"]
        }

        print(f"Status: success")
        print(f"Answer: {result['final_answer']}")
        print(f"Hops: {result['num_hops']}")
        print(f"Retrievals: {result['num_retrievals']}")
        print(f"LLM calls: {result['num_llm_calls']}")
        print(f"Time: {result['elapsed_seconds']} seconds")

    except Exception as e:
        elapsed_time = time.time() - start_time

        result = {
            "sample_index": i,
            "question": question,
            "status": "error",
            "error_type": type(e).__name__,
            "error_message": str(e),
            "elapsed_seconds": round(elapsed_time, 2)
        }

        print(f"Status: ERROR")
        print(f"Error: {type(e).__name__}: {e}")

    # 4. Save after every question, including failures
    debugging_results.append(result)

    with open(RESULTS_PATH, "w", encoding="utf-8") as f:
        json.dump(
            debugging_results,
            f,
            ensure_ascii=False,
            indent=2
        )

    print(f"Progress saved: {i}/{total_samples}")

# 5. Print a compact execution summary
successful = sum(
    r["status"] == "success"
    for r in debugging_results
)
failed = total_samples - successful

print("\n" + "=" * 50)
print("DEBUGGING EXPERIMENT FINISHED")
print(f"Total questions: {total_samples}")
print(f"Successful: {successful}")
print(f"Failed: {failed}")
print(f"Results file: {RESULTS_PATH}")

Starting debugging experiment
Total questions: 30
Max hops: 3
Top-k: 5
Results will be saved to: /content/drive/MyDrive/Multi_hop_RAG/data/hotpotqa/results/debugging_results_30_iterative_retrieval.json

[1/30] What was Iqbal F. Qadir on when he participated in an attack on a radar station located on western shore of the Okhamandal Peninsula?
Status: success
Answer: Cannot be determined from the evidence.
Hops: 3
Retrievals: 3
LLM calls: 6
Time: 38.34 seconds
Progress saved: 1/30

[2/30] When did the park at which Tivolis Koncertsal is located open?
Status: success
Answer: Tivoli Gardens, the park at which Tivolis Koncertsal is located, opened on 15 August 1843.
Hops: 1
Retrievals: 1
LLM calls: 2
Time: 13.26 seconds
Progress saved: 2/30

[3/30] What is the shared country of ancestry between Art Laboe and Scout Tufankjian?
Status: success
Answer: Armenia
Hops: 2
Retrievals: 2
LLM calls: 4
Time: 20.24 seconds
Progress saved: 3/30

[4/30] The school in which the Wilmslow Show is held is de

## 12. Summarize Results and Evaluate Answer Quality

Load the saved results, summarize execution statistics, and calculate Exact Match (EM) and token-level F1 against the reference answers. These metrics use normalized text and provide a simple comparison across methods.


In [ ]:
import json
import pandas as pd
from pathlib import Path

RESULTS_PATH = Path(
    RESULTS_DIR
) / DATASET_NAME / "debugging_results_30_iterative_retrieval.json"

with open(RESULTS_PATH, "r", encoding="utf-8") as f:
    debugging_results = json.load(f)

summary_df = pd.DataFrame([
    {
        "index": r["sample_index"],
        "status": r["status"],
        "question": r["question"],
        "final_answer": r.get("final_answer", ""),
        "hops": r.get("num_hops"),
        "retrievals": r.get("num_retrievals"),
        "llm_calls": r.get("num_llm_calls"),
        "time_seconds": r.get("elapsed_seconds"),
        "stop_reason": r.get("stop_reason")
    }
    for r in debugging_results
])

print("Total records:", len(summary_df))
print("\nExecution status:")
print(summary_df["status"].value_counts())

print("\nExecution statistics:")
print(
    summary_df[
        ["hops", "retrievals", "llm_calls", "time_seconds"]
    ].describe().round(2)
)

display(summary_df)

Total records: 30

Execution status:
status
success    30
Name: count, dtype: int64

Execution statistics:
        hops  retrievals  llm_calls  time_seconds
count  30.00       30.00      30.00         30.00
mean    1.87        1.87       3.73         22.88
std     0.90        0.90       1.80         10.76
min     1.00        1.00       2.00          8.21
25%     1.00        1.00       2.00         13.17
50%     2.00        2.00       4.00         19.88
75%     3.00        3.00       6.00         32.76
max     3.00        3.00       6.00         41.56


,index,status,question,final_answer,hops,retrievals,llm_calls,time_seconds,stop_reason
0,1,success,What was Iqbal F. Qadir on when he participate...,Cannot be determined from the evidence.,3,3,6,38.34,maximum_hops_reached
1,2,success,When did the park at which Tivolis Koncertsal ...,"Tivoli Gardens, the park at which Tivolis Konc...",1,1,2,13.26,sufficient_evidence
2,3,success,What is the shared country of ancestry between...,Armenia,2,2,4,20.24,sufficient_evidence
3,4,success,The school in which the Wilmslow Show is held ...,The school in which the Wilmslow Show is held ...,1,1,2,11.99,sufficient_evidence
4,5,success,Who will Billy Howle be seen opposite in the u...,Saoirse Ronan,1,1,2,13.90,sufficient_evidence
5,6,success,"What animated movie, starring Danny Devito, fe...",Cannot be determined from the evidence.,3,3,6,32.88,maximum_hops_reached
6,7,success,Out of the actors who have played the role of ...,Scott Adkins,1,1,2,15.78,sufficient_evidence
7,8,success,Tommy's Honour was a drama film that included ...,Tommy's Honour was a drama film that included ...,2,2,4,31.91,sufficient_evidence
8,9,success,Which rock band chose its name by drawing it o...,Midnight Oil chose its name by drawing it out ...,1,1,2,11.86,sufficient_evidence
9,10,success,"""Tunak"", is a bhangra/pop love song by an arti...",1967,2,2,4,19.53,sufficient_evidence


In [ ]:
import re
import string
import pandas as pd
from collections import Counter

def normalize_answer(text):
    """Normalize text for HotpotQA-style answer comparison."""
    text = text.lower()
    text = "".join(
        ch for ch in text
        if ch not in string.punctuation
    )
    text = re.sub(r"\b(a|an|the)\b", " ", text)
    text = " ".join(text.split())
    return text


def exact_match_score(prediction, reference):
    return int(
        normalize_answer(prediction)
        == normalize_answer(reference)
    )


def token_f1_score(prediction, reference):
    pred_tokens = normalize_answer(prediction).split()
    ref_tokens = normalize_answer(reference).split()

    if not pred_tokens or not ref_tokens:
        return float(pred_tokens == ref_tokens)

    common = Counter(pred_tokens) & Counter(ref_tokens)
    num_same = sum(common.values())

    if num_same == 0:
        return 0.0

    precision = num_same / len(pred_tokens)
    recall = num_same / len(ref_tokens)

    return 2 * precision * recall / (precision + recall)


# Match results and references by question ID when available,
# otherwise use the existing evaluation order.
evaluation_by_id = {
    sample["id"]: sample
    for sample in evaluation_samples
}

evaluation_by_question = {
    sample["question"]: sample
    for sample in evaluation_samples
}

comparison_rows = []

for result in debugging_results:
    sample = evaluation_by_question.get(result["question"])

    if sample is None:
        comparison_rows.append({
            "index": result["sample_index"],
            "question": result["question"],
            "reference_answer": None,
            "predicted_answer": result.get("final_answer", ""),
            "exact_match": None,
            "token_f1": None,
            "status": "reference_not_found"
        })
        continue

    if result["status"] != "success":
        comparison_rows.append({
            "index": result["sample_index"],
            "question": result["question"],
            "reference_answer": sample["answer"],
            "predicted_answer": "",
            "exact_match": 0,
            "token_f1": 0.0,
            "status": "execution_error"
        })
        continue

    prediction = result["final_answer"]
    reference = sample["answer"]

    comparison_rows.append({
        "index": result["sample_index"],
        "question": result["question"],
        "reference_answer": reference,
        "predicted_answer": prediction,
        "exact_match": exact_match_score(
            prediction, reference
        ),
        "token_f1": token_f1_score(
            prediction, reference
        ),
        "status": "evaluated"
    })

comparison_df = pd.DataFrame(comparison_rows)

valid_rows = comparison_df[
    comparison_df["status"] == "evaluated"
]

print("Total questions:", len(comparison_df))
print("Evaluated questions:", len(valid_rows))
print(
    "Exact Match:",
    f"{valid_rows['exact_match'].mean() * 100:.2f}%"
)
print(
    "Token F1:",
    f"{valid_rows['token_f1'].mean() * 100:.2f}%"
)

display(comparison_df)

Total questions: 30
Evaluated questions: 30
Exact Match: 30.00%
Token F1: 35.75%


,index,question,reference_answer,predicted_answer,exact_match,token_f1,status
0,1,What was Iqbal F. Qadir on when he participate...,flotilla,Cannot be determined from the evidence.,0,0.000000,evaluated
1,2,When did the park at which Tivolis Koncertsal ...,15 August 1843,"Tivoli Gardens, the park at which Tivolis Konc...",0,0.352941,evaluated
2,3,What is the shared country of ancestry between...,Armenian,Armenia,0,0.000000,evaluated
3,4,The school in which the Wilmslow Show is held ...,Centre of Excellence,The school in which the Wilmslow Show is held ...,0,0.375000,evaluated
4,5,Who will Billy Howle be seen opposite in the u...,Saoirse Ronan,Saoirse Ronan,1,1.000000,evaluated
5,6,"What animated movie, starring Danny Devito, fe...",The Lorax,Cannot be determined from the evidence.,0,0.000000,evaluated
6,7,Out of the actors who have played the role of ...,Scott Adkins,Scott Adkins,1,1.000000,evaluated
7,8,Tommy's Honour was a drama film that included ...,War & Peace,Tommy's Honour was a drama film that included ...,0,0.190476,evaluated
8,9,Which rock band chose its name by drawing it o...,Midnight Oil,Midnight Oil chose its name by drawing it out ...,0,0.307692,evaluated
9,10,"""Tunak"", is a bhangra/pop love song by an arti...",1967,1967,1,1.000000,evaluated


In [ ]:
# Identify questions whose predictions do not exactly match
# the reference answer.

failure_df = comparison_df[
    comparison_df["exact_match"] == 0
].copy()

print("Questions requiring review:", len(failure_df))
print("\nQuestions where the model abstained:")

abstained = failure_df[
    failure_df["predicted_answer"].str.contains(
        "cannot be determined",
        case=False,
        na=False
    )
]

print("Count:", len(abstained))

display(
    failure_df[
        [
            "index",
            "question",
            "reference_answer",
            "predicted_answer"
        ]
    ].reset_index(drop=True)
)

Questions requiring review: 21

Questions where the model abstained:
Count: 9


,index,question,reference_answer,predicted_answer
0,1,What was Iqbal F. Qadir on when he participate...,flotilla,Cannot be determined from the evidence.
1,2,When did the park at which Tivolis Koncertsal ...,15 August 1843,"Tivoli Gardens, the park at which Tivolis Konc..."
2,3,What is the shared country of ancestry between...,Armenian,Armenia
3,4,The school in which the Wilmslow Show is held ...,Centre of Excellence,The school in which the Wilmslow Show is held ...
4,6,"What animated movie, starring Danny Devito, fe...",The Lorax,Cannot be determined from the evidence.
5,8,Tommy's Honour was a drama film that included ...,War & Peace,Tommy's Honour was a drama film that included ...
6,9,Which rock band chose its name by drawing it o...,Midnight Oil,Midnight Oil chose its name by drawing it out ...
7,12,In which film did Emilio Estevez star in in th...,The Outsiders,"In 1983, Emilio Estevez starred in ""The Outsid..."
8,13,What was the concept of the business Eric S .P...,to ensure wide visibility and understanding of...,Cannot be determined from the evidence.
9,14,Which port city lies approximately 25 km north...,Keelung,Cannot be determined from the evidence.
